# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library and Python. All dataset entities are referenced by their `@id` as per FAIR best practices.

### Dataset Source
The dataset is defined by a Croissant schema accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load package metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset and extract its metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Inspect the record sets and their fields using their `@id` values.

In [ ]:
# List available record sets by @id
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    fields = rs.get('fields', [])
    print(f"  Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field.get('@id', 'N/A')} : {field.get('name', 'N/A')}")
        else:
            print(f"    - {field}")
    print()

## 3. Data Extraction
We'll load data from one or more record sets into Pandas DataFrames for further exploration and processing. **Please reference record sets and fields using their `@id`.**

In [ ]:
# Collect all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set '{record_set_id}'\n")

# If there are record sets loaded, display columns of the first one
if dataframes:
    chosen_record_set = record_set_ids[0]
    print(f"Columns in record set '{chosen_record_set}':\n", dataframes[chosen_record_set].columns.tolist())
    dataframes[chosen_record_set].head()
else:
    print("No record sets with data found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Process and analyze fields within a record set. We'll demonstrate by choosing a numeric field (by its `@id`), filtering for values above a threshold, normalizing it, and grouping by another field (also by `@id`). All identifiers used below should be replaced with actual `@id` values from the record set overview, if present.

In [ ]:
# Example: Choose a record set
if dataframes:
    record_set_id = chosen_record_set
    df = dataframes[record_set_id]

    # List all field @id's so the user knows what to select
    print("Available columns (by field @id):", df.columns.tolist())

    # For illustration, pick the first numeric field if any
    # (replace this with actual @id as per your schema overview)
    numeric_field_id = None
    for col in df.columns:
        # Test if column likely contains numeric data
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found for analysis.")
    else:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field for these rows
        normalized_col = numeric_field_id + "_normalized"
        filtered_df[normalized_col] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized '{numeric_field_id}':")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Find a grouping/categorical field to group by (not the numeric)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and pd.api.types.is_object_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped filtered data by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("\nNo suitable grouping field found.")
else:
    print("No dataframes loaded -- cannot perform EDA.")

## 5. Visualization
Now we visualize the distribution of the numeric field and grouped summaries, using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only visualize if EDA above was successful
if dataframes and 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' (Filtered > {threshold})")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouped DataFrame exists
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean of '{numeric_field_id}' by '{group_field_id}' (Filtered)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to load, explore, and analyze the FAIR^2 dataset, accessing all entities by their `@id` as required by FAIR and Croissant standards. 

We've:
- Loaded dataset metadata and record sets from a Croissant schema URL
- Inspected available record sets and their fields using `@id`
- Loaded tabular data from record sets for analysis
- Performed exploratory filtering and normalization on numeric fields using their `@id`
- Created basic visualizations of the data

For further processing, refer to the Croissant documentation and the full list of record and field `@id` values in this dataset.